# ROGII 04 — v6 library-free + linear prior (MD, Z)

**Hypothesis (08_v3_audit.md)**: v3's typewell library lookup caused LB 22.28 (over-fit on library leak; library = ALL train wells, doesn't generalise). v6 removes library entirely and adds hengck23's stage-1 idea: solve TVT *without GR* first via a per-well linear plane fit in (MD, Z) space.

**Pipeline**:
1. v1 features (anchor, trajectory, GR, single xcorr) — 43
2. v4 enhancements (multi-scale xcorr, jerk, gr_residual vs xcorr-shifted typewell) — 12
3. Row-level KNN imputation of formation_TVD with self-well LOO — 7
4. v6 linear prior: per-well TVT = a*MD + b*Z + c, fit on context, extrapolate to eval — 8
5. CatBoost 5-fold GroupKFold (well-CV) → submission

**Local OOF (well-CV)**: v1=14.81 → v6=14.23 (-0.58 ft).
**Predicted LB**: ~12.84 ft.


In [ ]:
import time, json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold

DATA_DIR = Path('/kaggle/input/rogii-wellbore-geology-prediction')
if not DATA_DIR.exists():
    DATA_DIR = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
print('DATA_DIR =', DATA_DIR)

GEO_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']
N_FOLDS = 5
SEED = 42


## 1. IO + partition utilities

In [ ]:
@dataclass
class WellPartition:
    well: str
    n_rows: int
    ps_idx: int
    ctx_idx: np.ndarray
    eval_idx: np.ndarray

def list_wells(kind):
    return sorted({p.name.split('__')[0] for p in (DATA_DIR / kind).glob('*__horizontal_well.csv')})

def load_horizontal(well, kind):
    return pd.read_csv(DATA_DIR / kind / f'{well}__horizontal_well.csv')

def load_typewell(well, kind):
    return pd.read_csv(DATA_DIR / kind / f'{well}__typewell.csv')

def partition_well(hw, well='?'):
    n = len(hw)
    nan_mask = hw['TVT_input'].isna().values
    if not nan_mask.any():
        return WellPartition(well, n, n, np.arange(n), np.array([], dtype=int))
    nan_idx = np.where(nan_mask)[0]
    ps_idx = int(nan_idx.min())
    return WellPartition(well, n, ps_idx, np.arange(0, ps_idx), nan_idx)

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

print(f'train wells: {len(list_wells("train"))}')
print(f'test wells:  {len(list_wells("test"))}')


## 2. CV: well-id GroupKFold (well-CV)

In [ ]:
def _shuffled_groupkfold(items, groups, n_splits, seed):
    n = len(items); rng = np.random.default_rng(seed); perm = rng.permutation(n)
    sh_items = [items[i] for i in perm]; sh_groups = [groups[i] for i in perm]
    gkf = GroupKFold(n_splits=n_splits)
    return [(perm[tr], perm[va]) for tr, va in gkf.split(sh_items, groups=sh_groups)]

def well_groupkfold(wells, n_splits=N_FOLDS, seed=SEED):
    wells = list(wells); return _shuffled_groupkfold(wells, wells, n_splits, seed)


## 3. v1 features (anchor + trajectory + GR + xcorr) — 43 features

In [ ]:
def _add_anchor_features(out, hw, part):
    eval_idx = part.eval_idx; ps = part.ps_idx
    md_eval = hw['MD'].iloc[eval_idx].values
    anchor = float(hw['TVT_input'].iloc[ps - 1]) if ps > 0 else float(hw['TVT_input'].dropna().iloc[0])
    out['anchor_tvt'] = np.full(len(eval_idx), anchor)
    md_anchor = float(hw['MD'].iloc[ps - 1])
    out['delta_md'] = md_eval - md_anchor
    out['log1p_delta_md'] = np.log1p(out['delta_md'])
    is_first = np.zeros(len(eval_idx), dtype=np.float32); is_first[0] = 1.0
    out['is_first_eval_row'] = is_first
    for w in (30, 100):
        start = max(0, ps - w)
        ctx_tvt = hw['TVT_input'].iloc[start:ps].values
        if len(ctx_tvt) >= 2:
            mean_v = float(ctx_tvt.mean()); std_v = float(ctx_tvt.std())
            md_ctx = hw['MD'].iloc[start:ps].values
            slope_v, _ = np.polyfit(md_ctx, ctx_tvt, 1); slope_v = float(slope_v)
        else:
            mean_v = anchor; std_v = 0.0; slope_v = 0.0
        out[f'ctx_tvt_mean_{w}'] = np.full(len(eval_idx), mean_v)
        out[f'ctx_tvt_std_{w}'] = np.full(len(eval_idx), std_v)
        out[f'ctx_tvt_slope_{w}'] = np.full(len(eval_idx), slope_v)
    out['anchor_minus_ctx_mean_30'] = out['anchor_tvt'] - out['ctx_tvt_mean_30']

def _add_trajectory_features(out, hw, part):
    eval_idx = part.eval_idx; ps = part.ps_idx
    z_eval = hw['Z'].iloc[eval_idx].values
    x_eval = hw['X'].iloc[eval_idx].values
    y_eval = hw['Y'].iloc[eval_idx].values
    z_anchor = float(hw['Z'].iloc[ps - 1])
    x_anchor = float(hw['X'].iloc[ps - 1]); y_anchor = float(hw['Y'].iloc[ps - 1])
    out['z'] = z_eval
    out['delta_z'] = z_eval - z_anchor
    out['delta_x'] = x_eval - x_anchor
    out['delta_y'] = y_eval - y_anchor
    out['delta_xy'] = np.sqrt(out['delta_x']**2 + out['delta_y']**2)
    md_full = hw['MD'].values; z_full = hw['Z'].values
    dz_dmd = np.zeros_like(md_full); dz_dmd[1:] = np.diff(z_full) / np.maximum(np.diff(md_full), 1e-9)
    out['dz_dmd'] = dz_dmd[eval_idx]
    z_smooth_slope = np.zeros_like(md_full); half_w = 5
    for i in range(half_w, len(md_full) - half_w):
        seg_md = md_full[i-half_w:i+half_w+1]; seg_z = z_full[i-half_w:i+half_w+1]
        if len(seg_md) >= 2 and seg_md[-1] != seg_md[0]:
            s, _ = np.polyfit(seg_md, seg_z, 1); z_smooth_slope[i] = s
    out['dz_dmd_smooth5'] = z_smooth_slope[eval_idx]
    start = max(0, ps - 30)
    if ps - start >= 2:
        ctx_md = hw['MD'].iloc[start:ps].values; ctx_z = hw['Z'].iloc[start:ps].values
        slope_z, _ = np.polyfit(ctx_md, ctx_z, 1)
        out['ctx_dz_dmd_30'] = np.full(len(eval_idx), float(slope_z))
    else:
        out['ctx_dz_dmd_30'] = np.zeros(len(eval_idx), dtype=np.float32)
    incl = np.full(len(md_full), 90.0, dtype=np.float32)
    if len(md_full) >= 2:
        dz = np.diff(z_full); dxy = np.sqrt(np.diff(hw['X'].values)**2 + np.diff(hw['Y'].values)**2)
        local_incl = np.degrees(np.arctan2(dxy, -dz + 1e-9))
        incl[1:] = local_incl
    out['inclination'] = incl[eval_idx]
    dogleg = np.zeros(len(md_full), dtype=np.float32)
    if len(md_full) >= 3:
        seg = np.diff(hw[['X','Y','Z']].values, axis=0)
        seg = seg / (np.linalg.norm(seg, axis=1, keepdims=True) + 1e-9)
        cos_th = np.sum(seg[:-1] * seg[1:], axis=1).clip(-1, 1)
        dogleg[2:] = np.degrees(np.arccos(cos_th))
    out['dogleg'] = dogleg[eval_idx]
    ctx_dogleg = dogleg[max(0, ps-100):ps]
    out['ctx_dogleg_p95'] = np.full(len(eval_idx), float(np.percentile(ctx_dogleg, 95)) if len(ctx_dogleg) else 0.0)
    out['abs_dz'] = np.abs(out['delta_z'])
    out['dz_per_dmd'] = out['delta_z'] / np.maximum(out['delta_md'], 1.0)

def _add_gr_features(out, hw, tw, part):
    eval_idx = part.eval_idx; ps = part.ps_idx
    gr_eval = hw['GR'].iloc[eval_idx].values
    out['gr'] = np.where(np.isnan(gr_eval), 0.0, gr_eval).astype(np.float32)
    out['gr_isna'] = np.isnan(gr_eval).astype(np.float32)
    full_gr = hw['GR'].values.copy()
    last_val = np.nan
    for i in range(len(full_gr)):
        if not np.isnan(full_gr[i]): last_val = full_gr[i]
        else: full_gr[i] = last_val if not np.isnan(last_val) else 0.0
    out['gr_ffill'] = full_gr[eval_idx]
    for w in (30, 100):
        start = max(0, ps - w)
        ctx_gr = hw['GR'].iloc[start:ps].dropna().values
        out[f'ctx_gr_mean_{w}'] = np.full(len(eval_idx), float(ctx_gr.mean()) if len(ctx_gr) else 0.0)
        out[f'ctx_gr_std_{w}']  = np.full(len(eval_idx), float(ctx_gr.std())  if len(ctx_gr) > 1 else 0.0)
    tw_clean = tw.dropna(subset=['GR']).sort_values('TVT')
    if len(tw_clean) >= 2:
        tw_tvt = tw_clean['TVT'].values; tw_gr = tw_clean['GR'].values
        anchor = float(out['anchor_tvt'][0])
        out['typewell_gr_at_anchor'] = np.full(len(eval_idx), float(np.interp(anchor, tw_tvt, tw_gr)))
        slope = float(out['ctx_tvt_slope_30'][0])
        pred_tvt = anchor + slope * out['delta_md']
        out['typewell_gr_at_pred_tvt'] = np.interp(pred_tvt, tw_tvt, tw_gr).astype(np.float32)
        out['gr_diff_typewell'] = (out['gr_ffill'] - out['typewell_gr_at_pred_tvt']).astype(np.float32)
    else:
        out['typewell_gr_at_anchor'] = np.zeros(len(eval_idx), dtype=np.float32)
        out['typewell_gr_at_pred_tvt'] = np.zeros(len(eval_idx), dtype=np.float32)
        out['gr_diff_typewell'] = np.zeros(len(eval_idx), dtype=np.float32)

def _add_well_meta(out, hw, tw, part):
    n = len(hw); eval_idx = part.eval_idx
    out['well_n_rows'] = np.full(len(eval_idx), float(n))
    out['well_eval_n'] = np.full(len(eval_idx), float(len(eval_idx)))
    out['well_eval_frac'] = np.full(len(eval_idx), len(eval_idx) / max(n, 1))
    out['well_gr_na_frac'] = np.full(len(eval_idx), float(hw['GR'].isna().mean()))
    out['well_tw_n'] = np.full(len(eval_idx), float(len(tw)))
    out['well_tw_tvt_range'] = np.full(len(eval_idx), float(tw['TVT'].max() - tw['TVT'].min()) if len(tw) else 0.0)

def _best_xcorr_shift(ctx_tvt, ctx_gr, tw_tvt, tw_gr, shifts):
    scores = np.zeros(len(shifts), dtype=np.float64)
    for i, s in enumerate(shifts):
        gr_lookup = np.interp(ctx_tvt + s, tw_tvt, tw_gr)
        scores[i] = float(np.mean((ctx_gr - gr_lookup) ** 2))
    best_idx = int(np.argmin(scores))
    best_shift = float(shifts[best_idx]); best_score = float(scores[best_idx])
    conf = 1.0 - (best_score / (scores.mean() + 1e-9))
    return best_shift, best_score, float(conf)

def _add_xcorr_features(out, hw, tw, part):
    eval_idx = part.eval_idx; ps = part.ps_idx
    tw_clean = tw.dropna(subset=['GR']).sort_values('TVT')
    if len(tw_clean) < 10:
        for c in ('xcorr_shift','xcorr_score','xcorr_confidence'):
            out[c] = np.zeros(len(eval_idx), dtype=np.float32)
        return
    tw_tvt = tw_clean['TVT'].values; tw_gr = tw_clean['GR'].values
    shifts = np.arange(-50, 51, 1, dtype=np.float64)
    start = max(0, ps - 30)
    ctx = hw.iloc[start:ps][['TVT_input','GR']].dropna()
    if len(ctx) >= 10:
        s, sc, c = _best_xcorr_shift(ctx['TVT_input'].values, ctx['GR'].values, tw_tvt, tw_gr, shifts)
    else:
        s, sc, c = 0.0, 1e6, 0.0
    out['xcorr_shift'] = np.full(len(eval_idx), s, dtype=np.float32)
    out['xcorr_score'] = np.full(len(eval_idx), sc, dtype=np.float32)
    out['xcorr_confidence'] = np.full(len(eval_idx), c, dtype=np.float32)

def build_v1_for_well(well, kind):
    hw = load_horizontal(well, kind); tw = load_typewell(well, kind)
    part = partition_well(hw, well)
    if len(part.eval_idx) == 0:
        return pd.DataFrame(), None, np.array([], dtype=int), hw, tw, part
    out = {}
    _add_anchor_features(out, hw, part)
    _add_trajectory_features(out, hw, part)
    _add_gr_features(out, hw, tw, part)
    _add_well_meta(out, hw, tw, part)
    _add_xcorr_features(out, hw, tw, part)
    X = pd.DataFrame(out); X['well'] = well
    if kind == 'train':
        anchor = float(out['anchor_tvt'][0])
        tvt_eval = hw['TVT'].iloc[part.eval_idx].values
        y = (tvt_eval - anchor).astype(np.float32)
    else:
        y = None
    return X, y, part.eval_idx, hw, tw, part


## 4. v4 enhancements: multi-scale xcorr + jerk + gr_residual (12 features)

In [ ]:
def _add_v4_extras(out, hw, tw, part):
    eval_idx = part.eval_idx; ps = part.ps_idx
    md_eval = hw['MD'].iloc[eval_idx].values
    md_anchor = float(hw['MD'].iloc[ps - 1])
    anchor = float(out['anchor_tvt'][0])

    tw_clean = tw.dropna(subset=['GR']).sort_values('TVT')
    new_cols = [
        'xcorr_shift_100', 'xcorr_shift_full', 'xcorr_shift_grad', 'xcorr_shift_std',
        'pred_tvt_slope100_minus_anchor', 'pred_tvt_slope_avg_minus_anchor',
        'gr_residual', 'gr_residual_roll5', 'gr_residual_cumsum', 'gr_residual_abs_p95',
        'jerk', 'jerk_smooth5',
    ]
    if len(tw_clean) < 10:
        for c in new_cols: out[c] = np.zeros(len(eval_idx), dtype=np.float32)
        return

    tw_tvt = tw_clean['TVT'].values; tw_gr = tw_clean['GR'].values
    shifts = np.arange(-50, 51, 1, dtype=np.float64)

    sh = {}
    for w in (30, 100):
        start = max(0, ps - w)
        ctx_df = hw.iloc[start:ps][['TVT_input','GR']].dropna()
        if len(ctx_df) >= 10:
            s, _, _ = _best_xcorr_shift(ctx_df['TVT_input'].values, ctx_df['GR'].values, tw_tvt, tw_gr, shifts)
        else:
            s = 0.0
        sh[w] = s
    ctx_full = hw.iloc[:ps][['TVT_input','GR']].dropna()
    if len(ctx_full) >= 10:
        sf, _, _ = _best_xcorr_shift(ctx_full['TVT_input'].values, ctx_full['GR'].values, tw_tvt, tw_gr, shifts)
    else:
        sf = sh.get(100, 0.0)

    s30 = sh[30]; s100 = sh[100]
    out['xcorr_shift_100']  = np.full(len(eval_idx), s100, dtype=np.float32)
    out['xcorr_shift_full'] = np.full(len(eval_idx), sf, dtype=np.float32)
    out['xcorr_shift_grad'] = np.full(len(eval_idx), sf - s30, dtype=np.float32)
    out['xcorr_shift_std']  = np.full(len(eval_idx), float(np.std([s30, s100, sf])), dtype=np.float32)

    slope_30 = float(out['ctx_tvt_slope_30'][0])
    slope_100 = float(out['ctx_tvt_slope_100'][0])
    slope_avg = 0.5 * (slope_30 + slope_100)
    delta_md = md_eval - md_anchor
    pred_tvt_100 = anchor + slope_100 * delta_md
    pred_tvt_avg = anchor + slope_avg * delta_md
    out['pred_tvt_slope100_minus_anchor'] = (pred_tvt_100 - anchor).astype(np.float32)
    out['pred_tvt_slope_avg_minus_anchor'] = (pred_tvt_avg - anchor).astype(np.float32)

    expected_gr = np.interp(pred_tvt_avg + sf, tw_tvt, tw_gr)
    residual = (out['gr_ffill'] - expected_gr).astype(np.float32)
    out['gr_residual'] = residual
    s_series = pd.Series(residual)
    out['gr_residual_roll5'] = s_series.rolling(5, min_periods=1).mean().values.astype(np.float32)
    cs = np.cumsum(residual) / np.sqrt(np.arange(1, len(residual)+1))
    out['gr_residual_cumsum'] = cs.astype(np.float32)
    p95 = float(np.percentile(np.abs(residual), 95)) if len(residual) else 0.0
    out['gr_residual_abs_p95'] = np.full(len(eval_idx), p95, dtype=np.float32)

    md_full = hw['MD'].values; z_full = hw['Z'].values
    n = len(md_full)
    dz = np.zeros(n); dz[1:] = np.diff(z_full) / np.maximum(np.diff(md_full), 1e-9)
    jerk = np.zeros(n)
    if n >= 3:
        jerk[1:] = np.diff(dz) / np.maximum(np.diff(md_full), 1e-9)
    out['jerk'] = jerk[eval_idx].astype(np.float32)
    jerk_smooth = pd.Series(jerk).rolling(5, min_periods=1, center=True).mean().values
    out['jerk_smooth5'] = jerk_smooth[eval_idx].astype(np.float32)


## 5. v6 linear prior in (MD, Z) (8 features) — hengck23 stage 1

In [ ]:
def _fit_plane(md, z, tvt):
    n = len(md)
    if n < 3:
        return 0.0, 0.0, float(np.median(tvt)) if n else 0.0, np.array([])
    A = np.column_stack([md, z, np.ones(n)])
    coef, *_ = np.linalg.lstsq(A, tvt, rcond=None)
    a, b, c = float(coef[0]), float(coef[1]), float(coef[2])
    resid = tvt - (a*md + b*z + c)
    return a, b, c, resid

def _add_v6_priors(out, hw, part):
    eval_idx = part.eval_idx; ps = part.ps_idx
    new_cols = ['prior_mdz_tvt','prior_mdz_minus_anchor','prior_mdz_residual_std',
                'prior_mdz_residual_p95','prior_mdz_a','prior_mdz_b','prior_mdz_c',
                'prior_mdz_extrapolation_dist']
    ctx = hw.iloc[:ps].dropna(subset=['TVT_input'])
    if len(ctx) < 5 or len(eval_idx) == 0:
        for c in new_cols: out[c] = np.zeros(len(eval_idx), dtype=np.float32)
        return
    md_ctx = ctx['MD'].to_numpy(dtype=np.float64)
    z_ctx = ctx['Z'].to_numpy(dtype=np.float64)
    tvt_ctx = ctx['TVT_input'].to_numpy(dtype=np.float64)
    a, b, c, resid = _fit_plane(md_ctx, z_ctx, tvt_ctx)
    resid_std = float(np.std(resid)) if len(resid) else 0.0
    resid_p95 = float(np.percentile(np.abs(resid), 95)) if len(resid) else 0.0
    md_eval = hw['MD'].iloc[eval_idx].to_numpy(dtype=np.float64)
    z_eval = hw['Z'].iloc[eval_idx].to_numpy(dtype=np.float64)
    tvt_pred = a*md_eval + b*z_eval + c
    anchor = float(out['anchor_tvt'][0])
    md_centroid = md_ctx.mean(); z_centroid = z_ctx.mean()
    extrap_dist = np.sqrt((md_eval - md_centroid)**2 + (z_eval - z_centroid)**2)
    out['prior_mdz_tvt'] = tvt_pred.astype(np.float32)
    out['prior_mdz_minus_anchor'] = (tvt_pred - anchor).astype(np.float32)
    out['prior_mdz_residual_std'] = np.full(len(eval_idx), resid_std, dtype=np.float32)
    out['prior_mdz_residual_p95'] = np.full(len(eval_idx), resid_p95, dtype=np.float32)
    out['prior_mdz_a'] = np.full(len(eval_idx), a, dtype=np.float32)
    out['prior_mdz_b'] = np.full(len(eval_idx), b, dtype=np.float32)
    out['prior_mdz_c'] = np.full(len(eval_idx), c, dtype=np.float32)
    out['prior_mdz_extrapolation_dist'] = extrap_dist.astype(np.float32)


## 6. Build v1 + v4 + v6 features (62 features so far)

In [ ]:
def build_features_for_well(well, kind):
    X, y, eidx, hw, tw, part = build_v1_for_well(well, kind)
    if len(X) == 0: return X, y, eidx
    # We need out dict to add v4 + v6 features. Re-build via fresh out:
    out = {c: X[c].to_numpy() for c in X.columns if c != 'well'}
    _add_v4_extras(out, hw, tw, part)
    _add_v6_priors(out, hw, part)
    Xf = pd.DataFrame(out); Xf['well'] = well
    return Xf, y, eidx

def build_features(kind, verbose=True):
    wells = list_wells(kind)
    pieces_X = []; pieces_y = []; well_eval_idx = {}
    t0 = time.time()
    for i, w in enumerate(wells):
        X, y, eidx = build_features_for_well(w, kind)
        if len(X) == 0: continue
        pieces_X.append(X)
        if y is not None: pieces_y.append(y)
        well_eval_idx[w] = eidx
        if verbose and (i+1) % 100 == 0:
            print(f'  {i+1}/{len(wells)} ({time.time()-t0:.1f}s)', flush=True)
    X_all = pd.concat(pieces_X, ignore_index=True)
    y_all = np.concatenate(pieces_y) if pieces_y else None
    return X_all, y_all, well_eval_idx

print('building features (train)...')
t0 = time.time()
X_train, y_train, train_eidx = build_features('train')
print(f'  train: {X_train.shape}  ({time.time()-t0:.0f}s)')

print('building features (test)...')
t1 = time.time()
X_test, _, test_eval_idx = build_features('test')
print(f'  test:  {X_test.shape}  ({time.time()-t1:.1f}s)')


## 7. Row-level KNN imputation of formation_TVD (7 features, self-well LOO only)

In [ ]:
print('loading all train rows for cKDTree...')
t0 = time.time()
hw_files = sorted((DATA_DIR / 'train').glob('*__horizontal_well.csv'))
parts = []
for p in hw_files:
    wid = p.name.split('__')[0]
    df = pd.read_csv(p, usecols=['X','Y','Z', *GEO_COLS]); df['well_id'] = wid
    parts.append(df)
all_train = pd.concat(parts, ignore_index=True).dropna(subset=GEO_COLS)
print(f'  {len(all_train):,} rows ({time.time()-t0:.1f}s)')

train_xy = all_train[['X','Y']].to_numpy()
train_wells_arr = all_train['well_id'].to_numpy()
train_form_arr = all_train[GEO_COLS].to_numpy().astype(np.float32)
print('building cKDTree...')
tree = cKDTree(train_xy)
print(f'  tree built ({time.time()-t0:.1f}s total)')

def collect_eval_xy(X, kind):
    eval_xy = np.zeros((len(X), 2), dtype=np.float64)
    for w in X['well'].unique():
        m = (X['well'] == w).to_numpy()
        hw = load_horizontal(w, kind)
        eidx = np.where(hw['TVT_input'].isna().values)[0]
        n_w = int(m.sum())
        xy = hw[['X','Y']].iloc[eidx[:n_w]].to_numpy()
        eval_xy[m] = xy
    return eval_xy

def knn_form_pred(eval_xy, eval_well, k_query=40, k_keep=20):
    dist, idx = tree.query(eval_xy, k=k_query, workers=-1)
    nn_wells = train_wells_arr[idx]
    self_mask = nn_wells == eval_well[:, None]
    dist_eff = np.where(self_mask, np.inf, dist)
    order = np.argsort(dist_eff, axis=1, kind='stable')
    keep_cols = order[:, :k_keep]
    rows_arr = np.arange(len(eval_xy))[:, None]
    keep_dist = dist[rows_arr, keep_cols]
    keep_idx_train = idx[rows_arr, keep_cols]
    w = 1.0 / (keep_dist + 1e-6); w /= w.sum(axis=1, keepdims=True)
    f_kept = train_form_arr[keep_idx_train]
    pred_form = (f_kept * w[:, :, None]).sum(axis=1).astype(np.float32)
    pred_std = f_kept.std(axis=1).mean(axis=1).astype(np.float32)
    return pred_form, pred_std

def add_knn_features(X, kind):
    eval_xy = collect_eval_xy(X, kind)
    eval_well = X['well'].to_numpy()
    pred_form, pred_std = knn_form_pred(eval_xy, eval_well)
    for i, g in enumerate(GEO_COLS):
        X[f'form_{g}_knn'] = pred_form[:, i]
    X['form_knn_std'] = pred_std
    return X

print('adding KNN form features (train)...')
t0 = time.time()
X_train = add_knn_features(X_train, 'train')
print(f'  train shape: {X_train.shape}  ({time.time()-t0:.1f}s)')

print('adding KNN form features (test)...')
t1 = time.time()
X_test = add_knn_features(X_test, 'test')
print(f'  test shape: {X_test.shape}  ({time.time()-t1:.1f}s)')


## 8. CatBoost 5-fold well-CV training

In [ ]:
from catboost import CatBoostRegressor

feat_cols = [c for c in X_train.columns if c != 'well']
train_wells = sorted(X_train['well'].unique())
print(f'#features: {len(feat_cols)}  #train wells: {len(train_wells)}')

splits = well_groupkfold(train_wells)
well_to_pos = {w: i for i, w in enumerate(train_wells)}
row_well_pos = X_train['well'].map(well_to_pos).values

oof_pred = np.zeros(len(X_train), dtype=np.float32)
test_pred_per_fold = np.zeros((N_FOLDS, len(X_test)), dtype=np.float32)
t0 = time.time()
for fold, (tr_pos, va_pos) in enumerate(splits):
    tr_mask = np.isin(row_well_pos, tr_pos)
    va_mask = np.isin(row_well_pos, va_pos)
    n_tr_w = len({train_wells[i] for i in tr_pos}); n_va_w = len({train_wells[i] for i in va_pos})
    print(f'\n[fold {fold+1}/{N_FOLDS}] train_rows={tr_mask.sum():,} ({n_tr_w} wells)  val_rows={va_mask.sum():,} ({n_va_w} wells)')
    Xt = X_train.loc[tr_mask, feat_cols]; Xv = X_train.loc[va_mask, feat_cols]
    yt = y_train[tr_mask]; yv = y_train[va_mask]
    model = CatBoostRegressor(
        iterations=8000, learning_rate=0.05, depth=8,
        loss_function='RMSE', eval_metric='RMSE',
        random_seed=42, od_type='Iter', od_wait=200,
        task_type='GPU', verbose=400, allow_writing_files=False,
    )
    model.fit(Xt, yt, eval_set=(Xv, yv), use_best_model=True)
    oof_pred[va_mask] = model.predict(Xv)
    test_pred_per_fold[fold] = model.predict(X_test[feat_cols])
    rmse_va = float(np.sqrt(np.mean((yv - oof_pred[va_mask])**2)))
    print(f'  fold {fold+1} val RMSE = {rmse_va:.4f}  best_iter={model.best_iteration_}')

oof_rmse = rmse(y_train, oof_pred)
print(f'\n[done] OOF pooled RMSE = {oof_rmse:.4f}  (elapsed {time.time()-t0:.0f}s)')


## 9. Build submission

In [ ]:
test_pred_delta = test_pred_per_fold.mean(axis=0)
test_pred_tvt = X_test['anchor_tvt'].values + test_pred_delta

sub_rows = []
for w in X_test['well'].unique():
    m = (X_test['well'] == w).to_numpy()
    for ridx, p in zip(test_eval_idx[w], test_pred_tvt[m]):
        sub_rows.append({'id': f'{w}_{int(ridx)}', 'tvt': float(p)})
preds = pd.DataFrame(sub_rows)
print('preds:', preds.shape)

ss = pd.read_csv(DATA_DIR / 'sample_submission.csv')
submission = ss[['id']].merge(preds, on='id', how='left')
n_missing = submission['tvt'].isna().sum()
if n_missing > 0:
    print(f'WARNING {n_missing} missing — filling with mean')
    submission['tvt'] = submission['tvt'].fillna(preds['tvt'].mean())
submission.to_csv('submission.csv', index=False)
print('submission.csv shape:', submission.shape)
print('tvt stats:', submission['tvt'].describe().round(3).to_dict())
